## Einsum

In [ ]:
"""

"""

## Matmul Outer

In [ ]:
import numpy as np
import neuronxcc.nki as nki
import neuronxcc.nki.isa as ni
import neuronxcc.nki.language as nl

@nki.jit
def matmul_outer_diff(AH, BH, CH, M1, M0, N2, N1):
    ZH = nl.ndarray((*[N2, N1, 128], *[M1, M0]), dtype=nl.float16, buffer=nl.shared_hbm)
    for as_n2 in nl.affine_range(N2):
        TS = nl.zeros((nl.par_dim(*[128]), *[N1]), dtype=nl.float16, buffer=nl.sbuf)
        for as_m1 in nl.affine_range(M1):
            AS = nl.ndarray((nl.par_dim(*[128]), *[M0 // 128, 1]), dtype=nl.float16, buffer=nl.sbuf)
            for as_m02 in nl.affine_range(M0 // 128):
                AS[:, as_m02, :] = nl.load(AH[as_m1, as_m02, :, :])
            # Edit: Flatten N1 and N0
            BS = nl.ndarray((nl.par_dim(*[128]), *[M0 // 128, N1 * 128]), dtype=nl.float16, buffer=nl.sbuf)
            for bs_m01 in nl.affine_range(M0 // 128):
                for bs_n1 in nl.affine_range(N1):
                    # Edit: Flatten N1 and N0
                    BS[:, bs_m01, :] = nl.load(BH[as_m1, bs_m01, :, as_n2, :])
            TP = nl.zeros((nl.par_dim(*[128]), *[N1]), dtype=nl.float32, buffer=nl.psum)
            for tp_n1 in nl.affine_range(N1):
                for tp_m01 in nl.affine_range(M0 // 128):
                    AS_for_TP = AS.reshape((128, M0 // 128))
                    # Edit: Unflatten for TP
                    BS_for_TP = BS.reshape((128, M0 // 128, N1, 128))
                    TP[:, tp_n1] += ni.nc_matmul(BS_for_TP[:, tp_m01, tp_n1, :], AS_for_TP[:, tp_m01])
                TS[:, tp_n1] = nl.loop_reduce(TP[:, tp_n1], np.add, loop_indices=[as_m1], dtype=nl.float16)
        for cs_m1 in nl.affine_range(M1):
            CS = nl.ndarray((nl.par_dim(*[1]), *[M0 // 512, 512]), dtype=nl.float16, buffer=nl.sbuf)
            for cs_m02 in nl.affine_range(M0 // 512):
                CS[:, cs_m02, :] = nl.load(CH[cs_m1, cs_m02, :, :])
            DP = nl.zeros((nl.par_dim(*[128]), *[N1, M0 // 128, 128]), dtype=nl.float32, buffer=nl.psum)
            DS = nl.ndarray((nl.par_dim(*[128]), *[N1, M0]), dtype=nl.float16, buffer=nl.sbuf)
            for dp_n1 in nl.affine_range(N1):
                for dp_m01 in nl.affine_range(M0 // 128):
                    # Edit: Unflatten for DP
                    BS_for_DP = BS.reshape((128, M0 // 128, N1, 128))
                    DP[:, dp_n1, dp_m01, :] = ni.nc_transpose(BS_for_DP[:, dp_m01, dp_n1, :])
                DP_for_DS = DP.reshape((128, N1, M0))
                DS[:, dp_n1, :] = nl.copy(DP_for_DS[:, dp_n1, :], dtype=nl.float16)
            CTS = nl.ndarray((nl.par_dim(*[128]), *[M0 // 512, 512, 1]), dtype=nl.float16, buffer=nl.sbuf)
            for cts_m02 in nl.affine_range(M0 // 512):
                CTS[:, cts_m02, :, :] = nl.broadcast_to(CS[:, cts_m02, :], shape=tuple([1, 128, 512][1:]))
            for ks_n1 in nl.affine_range(N1):
                KS = nl.ndarray((nl.par_dim(*[128]), *[M0]), dtype=nl.float16, buffer=nl.sbuf)
                CTS_for_KS = CTS.reshape((128, M0))
                KS[:, :] = ni.tensor_scalar(CTS_for_KS[:, :], np.multiply, TS[:, ks_n1])
                ZS = nl.ndarray((nl.par_dim(*[128]), *[M0]), dtype=nl.float16, buffer=nl.sbuf)
                ZS[:, :] = ni.tensor_tensor(DS[:, ks_n1, :], KS[:, :], np.subtract)
                nl.store(ZH[as_n2, ks_n1, :, cs_m1, :], value=ZS[:, :])
    return ZH

## Test

In [ ]:
from neuronxcc import nki
import numpy as np
from matmul_outer_diff import matmul_outer_diff

M = 12288
N = 22528

a = np.random.rand(M,1).astype(np.float16)
r = np.random.rand(N,M).astype(np.float16)

A = a.reshape(6, 16, 128, 1)
B = b.reshape(6, 16, 128, 11, 16 * 128)
C = a.reshape(6, 4, 1, 512)
D = d.reshape(11, 16, 128, 6, 2048)

expected = r - a @ (a.T @ r)
result_nki = matmul_outer_diff(A, B, C, D, M1=6, M0=2048, N2=11, N1=16, K1=6, K0=2048)
result_nki = result_nki.reshape(22528, 12288).T
print(expected)
print(result_nki)
is_close = np.allclose(result_nki, expected, rtol=1e-2, atol=1e-3)
print("Result match: ", is_close)

## Benchmark

In [ ]:
import neuronxcc.nki as nki
import numpy as np
from matmul_outer_diff import matmul_outer_diff

M = 12288
N = 22528

a = np.random.rand(M,1).astype(np.float16)
b = np.random.rand(M,N).astype(np.float16)
d = np.random.rand(N,M).astype(np.float16)

A = a.reshape(6, 16, 128, 1)
B = b.reshape(6, 16, 128, 11, 16 * 128)
C = a.reshape(6, 4, 1, 512)
D = d.reshape(11, 16, 128, 6, 2048)

def benchmark_nki(nki_func):
    bench_func = nki.benchmark(warmup=5, iters=10)(nki_func)
    bench_func(A, B, C, D, M1=6, M0=2048, N2=11, N1=16, K1=6, K0=2048)
    latency_res = bench_func.benchmark_result.nc_latency
    p99 = latency_res.get_latency_percentile(99)
    print("Latency: {:.2f} ms (P99)".format(p99 / 1000.0))

print("Benchmarking matmul_outer_diff")
benchmark_nki(matmul_outer_diff)

## Profile

In [ ]:
from neuronxcc import nki
from pathlib import Path
import numpy as np
from matmul_outer_diff import matmul_outer_diff

WORKING_DIRECTORY = Path.cwd()

M = 12288
N = 22528

a = np.random.rand(M,1).astype(np.float16)
b = np.random.rand(M,N).astype(np.float16)
d = np.random.rand(N,M).astype(np.float16)

A = a.reshape(6, 16, 128, 1)
B = b.reshape(6, 16, 128, 11, 16 * 128)
C = a.reshape(6, 4, 1, 512)
D = d.reshape(11, 16, 128, 6, 2048)

profile_func = nki.profile(working_directory=WORKING_DIRECTORY, save_neff_name='file.neff', 
                           save_trace_name='profile.ntff', profile_nth=2)(matmul_outer_diff)
profile_func(A, B, C, D, M1=6, M0=2048, N2=11, N1=16, K1=6, K0=2048)